# PIPELINE TIỀN XỬ LÝ DỮ LIỆU YOLO26 (TỪ YOLO DATASET)
Pipeline gồm ba bước:
0. **ĐỔI JSON** — chuyển bản export COCO của Roboflow sang định dạng YOLO (bỏ qua nếu đã có sẵn thư mục YOLO).
1. **SPLIT trước** — chia 70/20/10 trên ảnh gốc (val/test chỉ chứa ảnh gốc).
2. **AUGMENT sau** — chỉ tăng cường ảnh trong tập train.

> ⚠️ Thứ tự này là **bắt buộc để chống data leak**: nếu augment trước rồi mới chia, ảnh ảo của một ảnh val/test sẽ lọt vào train và làm sai lệch (thổi phồng) kết quả đánh giá.

In [1]:
import os
import cv2
import random
import shutil
import numpy as np
from tqdm import tqdm
import albumentations as A

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN TỔNG
# ==========================================
BASE_DIR = 'E:/TIEN_XU_LI_ANH_ANTENNA'

# Đầu vào: Thư mục chứa tập YOLO bạn vừa export
INPUT_IMG_DIR = os.path.join(BASE_DIR, 'Nha_tram.yolo26', 'train', 'images')
INPUT_LBL_DIR = os.path.join(BASE_DIR, 'Nha_tram.yolo26', 'train', 'labels')

# Đầu ra: Thư mục lưu kết quả cuối cùng đã chia tập
OUTPUT_DIR = os.path.join(BASE_DIR, 'Output_files')
FINAL_SPLIT_DIR = os.path.join(OUTPUT_DIR, 'YOLO_Final_Dataset')

CLASSES = ['anten-4G', 'anten-5G', 'none', 'rrh', 'rru', 'viba']
VIBA_CLASS_ID = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Cấu hình đường dẫn hoàn tất!")
print("⚙️ Thứ tự pipeline: SPLIT trước -> AUGMENT chỉ trên tập TRAIN (chống data leak)")

c:\Users\quanh\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Cấu hình đường dẫn hoàn tất!
⚙️ Thứ tự pipeline: SPLIT trước -> AUGMENT chỉ trên tập TRAIN (chống data leak)


### BƯỚC 0: ĐỔI COCO JSON SANG ĐỊNH DẠNG YOLO
Dữ liệu gốc được gán nhãn trên Roboflow và xuất ra định dạng **COCO** (một tệp `_annotations.coco.json`
chứa toàn bộ khung bao của mọi ảnh). YOLO lại cần mỗi ảnh một tệp `.txt` riêng, nên cần bước chuyển đổi này.

Hai khác biệt về định dạng phải xử lý:

| | COCO | YOLO |
|---|---|---|
| Cách ghi khung bao | `[x_góc_trái, y_góc_trên, rộng, cao]` | `[x_tâm, y_tâm, rộng, cao]` |
| Đơn vị | điểm ảnh tuyệt đối | chuẩn hoá về 0–1 theo cỡ ảnh |
| Tổ chức tệp | một tệp JSON cho cả tập | một tệp `.txt` cho mỗi ảnh |

Ngoài ra, bản export COCO của Roboflow luôn chèn thêm một category cha (ở đây là `Nha-tram`, id 0) không
phải lớp đối tượng thật. Cell dưới ghép lớp **theo tên** với danh sách `CLASSES` nên category cha tự bị
loại, đồng thời bảo đảm chỉ số lớp khớp đúng với `dataset.yaml` sinh ra ở BƯỚC 2.

> **Lưu ý khi chạy:** bộ dữ liệu YOLO đã có sẵn, nên cell được đặt `GHI_DE = False` — nó chỉ kiểm tra và
> báo cáo, không ghi đè gì. Muốn tạo lại từ đầu thì đổi thành `GHI_DE = True`.

In [ ]:
# ==========================================
# BƯỚC 0: ĐỔI COCO JSON -> ĐỊNH DẠNG YOLO
# ==========================================
import json, glob, collections

COCO_DIR = os.path.join(BASE_DIR, 'Nha_tram.v2-final_data.coco', 'train')   # ảnh + tệp _annotations.coco.json
YOLO_OUT = os.path.join(BASE_DIR, 'Nha_tram.yolo26', 'train')               # = INPUT_IMG_DIR / INPUT_LBL_DIR

GHI_DE = False   # False = chỉ kiểm tra, KHÔNG đụng vào dữ liệu đã có
                 # True  = xoá YOLO_OUT rồi chuyển đổi lại từ đầu


def doi_coco_sang_yolo(coco_dir, yolo_out, classes, ghi_de=False):
    """Đổi một bản export COCO sang định dạng YOLO. Trả về bảng đếm đối tượng theo lớp."""
    json_files = glob.glob(os.path.join(coco_dir, '*.json'))
    assert json_files, f'Không thấy tệp .json nào trong {coco_dir}'
    with open(json_files[0], encoding='utf-8') as f:
        coco = json.load(f)
    print(f'Đọc: {os.path.basename(json_files[0])}')
    print(f'  {len(coco["images"])} ảnh · {len(coco["annotations"])} khung bao')

    # --- Ánh xạ category_id -> chỉ số lớp YOLO, ghép THEO TÊN để chắc chắn khớp CLASSES ---
    id2cls = {c['id']: classes.index(c['name']) for c in coco['categories'] if c['name'] in classes}
    bo_qua = [c['name'] for c in coco['categories'] if c['name'] not in classes]
    print('  Ánh xạ lớp:', {c['name']: id2cls[c['id']] for c in coco['categories'] if c['id'] in id2cls})
    if bo_qua:
        print('  Bỏ category không phải lớp thật:', bo_qua)
    assert len(id2cls) == len(classes), 'Có lớp trong CLASSES không tìm thấy trong tệp JSON!'

    anns = collections.defaultdict(list)
    for a in coco['annotations']:
        anns[a['image_id']].append(a)

    img_out, lbl_out = os.path.join(yolo_out, 'images'), os.path.join(yolo_out, 'labels')
    if ghi_de and os.path.exists(yolo_out):
        shutil.rmtree(yolo_out)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    dem, anh_nen, box_loi, thieu = collections.Counter(), 0, 0, []
    for im in tqdm(coco['images'], desc='  Đổi COCO -> YOLO'):
        src = os.path.join(coco_dir, im['file_name'])
        if not os.path.exists(src):
            thieu.append(im['file_name'])
            continue
        W, H = float(im['width']), float(im['height'])
        dong = []
        for a in anns.get(im['id'], []):
            if a['category_id'] not in id2cls or a.get('iscrowd', 0):
                continue
            x, y, w, h = a['bbox']
            xc, yc = (x + w / 2) / W, (y + h / 2) / H      # góc trên-trái -> tâm
            wn, hn = w / W, h / H                          # tuyệt đối -> chuẩn hoá
            xc, yc = min(max(xc, 0.0), 1.0), min(max(yc, 0.0), 1.0)
            wn, hn = min(max(wn, 0.0), 1.0), min(max(hn, 0.0), 1.0)
            if wn <= 1e-6 or hn <= 1e-6:                   # khung suy biến -> bỏ
                box_loi += 1
                continue
            cls = id2cls[a['category_id']]
            dem[classes[cls]] += 1
            dong.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
        if not dong:
            anh_nen += 1
        shutil.copy2(src, os.path.join(img_out, im['file_name']))
        with open(os.path.join(lbl_out, os.path.splitext(im['file_name'])[0] + '.txt'),
                  'w', encoding='utf-8') as f:
            f.write('\n'.join(dong))

    print(f'\n  Đã ghi {len(os.listdir(img_out))} ảnh và {len(os.listdir(lbl_out))} tệp nhãn')
    if anh_nen:
        print(f'  Ảnh nền (không chứa đối tượng): {anh_nen}')
    if box_loi:
        print(f'  [!] Bỏ {box_loi} khung bao suy biến')
    if thieu:
        print(f'  [!] Thiếu {len(thieu)} ảnh so với tệp JSON, ví dụ: {thieu[:3]}')
    return dem


def dem_nhan_yolo(lbl_dir, classes):
    """Đếm đối tượng theo lớp trong một thư mục nhãn YOLO."""
    dem = collections.Counter()
    for f in glob.glob(os.path.join(lbl_dir, '*.txt')):
        with open(f, encoding='utf-8') as fh:
            for ln in fh:
                if ln.strip():
                    dem[classes[int(ln.split()[0])]] += 1
    return dem


# ------------------------------------------------------------------
lbl_dir = os.path.join(YOLO_OUT, 'labels')
da_co = os.path.isdir(lbl_dir) and len(os.listdir(lbl_dir)) > 0

if da_co and not GHI_DE:
    print('Bộ dữ liệu YOLO đã có sẵn -> BỎ QUA bước đổi định dạng, không ghi đè gì.')
    print(f'  {YOLO_OUT}')
    dem = dem_nhan_yolo(lbl_dir, CLASSES)
else:
    dem = doi_coco_sang_yolo(COCO_DIR, YOLO_OUT, CLASSES, ghi_de=GHI_DE)

print(f'\nSố ảnh  : {len(os.listdir(os.path.join(YOLO_OUT, "images")))}')
print(f'Số nhãn : {len(os.listdir(lbl_dir))}')
print(f'Tổng đối tượng: {sum(dem.values())}')
for c in CLASSES:
    print(f'   {c:10} {dem[c]:6}')
print('\n=> Dữ liệu sẵn sàng ở', YOLO_OUT, '- chạy tiếp BƯỚC 1.')

### BƯỚC 1: CHIA TẬP DATASET 7:2:1 (TRÊN ẢNH GỐC — CHỐNG LEAKAGE)
**Chia tập TRƯỚC khi augment.** Chỉ chia trên ảnh gốc, tách 70% Train / 20% Val / 10% Test.
Class `viba` (ít mẫu) được chia riêng theo đúng tỉ lệ để cân bằng. Val/Test chỉ chứa ảnh gốc → ảnh ảo không bao giờ lọt sang Val/Test.

In [2]:
print("\n--- BƯỚC 1: CHIA TẬP DATASET 7:2:1 (TRÊN ẢNH GỐC) ---")

# Dọn sạch thư mục output cũ (xóa dữ liệu RÒ RỈ từ lần chạy trước, nếu có)
if os.path.exists(FINAL_SPLIT_DIR):
    shutil.rmtree(FINAL_SPLIT_DIR)
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(FINAL_SPLIT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(FINAL_SPLIT_DIR, split, 'labels'), exist_ok=True)


def get_file_info(img_path, lbl_path):  # quét nhãn để biết ảnh nào chứa class viba
    files = []
    valid_ext = ('.jpg', '.jpeg', '.png')
    if not os.path.exists(img_path):
        return files
    for f in os.listdir(img_path):
        if f.lower().endswith(valid_ext):
            name = os.path.splitext(f)[0]
            txt_file = os.path.join(lbl_path, name + '.txt')
            has_viba = False
            if os.path.exists(txt_file):
                with open(txt_file, 'r') as t:
                    for line in t:
                        if line.startswith(str(VIBA_CLASS_ID) + " "):
                            has_viba = True
                            break
            files.append({'name': name, 'ext': os.path.splitext(f)[1], 'has_viba': has_viba})
    return files


# Chỉ làm việc trên ẢNH GỐC (chưa augment)
all_files = get_file_info(INPUT_IMG_DIR, INPUT_LBL_DIR)
orig_viba = [f for f in all_files if f['has_viba']]      # nhóm ảnh có viba (ít) -> chia riêng cho cân bằng
orig_others = [f for f in all_files if not f['has_viba']]  # nhóm ảnh còn lại

random.seed(42)  # cố định để mỗi lần chạy chia giống hệt nhau, 1 ảnh không nhảy tập
random.shuffle(orig_viba)
random.shuffle(orig_others)

train_set, val_set, test_set = [], [], []


def split_3way(items, tr, va, te):  # chia 70/20/10 cho từng nhóm
    n = len(items)
    n_tr = int(n * 0.7)
    n_val = int(n * 0.7) + int(n * 0.2)
    tr.extend(items[:n_tr])
    va.extend(items[n_tr:n_val])
    te.extend(items[n_val:])


split_3way(orig_viba, train_set, val_set, test_set)
split_3way(orig_others, train_set, val_set, test_set)


def copy_files(file_list, split_name):  # copy ảnh gốc + nhãn vào đúng tập
    for f in tqdm(file_list, desc=f"Copy ảnh gốc -> {split_name}"):
        full_img = f['name'] + f['ext']
        full_lbl = f['name'] + '.txt'
        shutil.copy(os.path.join(INPUT_IMG_DIR, full_img),
                    os.path.join(FINAL_SPLIT_DIR, split_name, 'images', full_img))
        src_lbl = os.path.join(INPUT_LBL_DIR, full_lbl)
        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, os.path.join(FINAL_SPLIT_DIR, split_name, 'labels', full_lbl))


copy_files(train_set, 'train')
copy_files(val_set, 'val')
copy_files(test_set, 'test')

print(f"✅ Đã chia ảnh GỐC: Train ({len(train_set)}) - Val ({len(val_set)}) - Test ({len(test_set)})")
print("⚠️ Val/Test CHỈ chứa ảnh gốc, chưa augment -> không leak.")


--- BƯỚC 1: CHIA TẬP DATASET 7:2:1 (TRÊN ẢNH GỐC) ---


Copy ảnh gốc -> test: 100%|██████████| 206/206 [00:02<00:00, 88.04it/s] 

✅ Đã chia ảnh GỐC: Train (1423) - Val (406) - Test (206)
⚠️ Val/Test CHỈ chứa ảnh gốc, chưa augment -> không leak.


### BƯỚC 2: DATA AUGMENTATION (CHỈ TRÊN TẬP TRAIN)
Sinh thêm biến thể (Grayscale, Lật, Xoay, Nhiễu...) **chỉ cho ảnh trong tập train**. Mỗi ảnh gốc đẻ thêm 2 ảnh ảo, ghi thẳng vào `train/images`.
Vì val/test đã tách ra từ BƯỚC 1 nên ảnh ảo của chúng không tồn tại → **không còn data leak**. Cuối cùng tạo `dataset.yaml` để train.

In [3]:
print("\n--- BƯỚC 2: DATA AUGMENTATION (CHỈ TRÊN TẬP TRAIN) ---")

TRAIN_IMG_DIR = os.path.join(FINAL_SPLIT_DIR, 'train', 'images')
TRAIN_LBL_DIR = os.path.join(FINAL_SPLIT_DIR, 'train', 'labels')

bbox_params = A.BboxParams(format='yolo', label_fields=['class_labels'], clip=True, min_visibility=0.1)

transform_gray = A.Compose([
    A.Resize(height=640, width=640, p=1.0),
    A.ToGray(p=1.0)
], bbox_params=bbox_params)

transform_random = A.Compose([
    A.Resize(height=640, width=640, p=1.0),
    A.SomeOf([
        A.HorizontalFlip(p=1),
        A.VerticalFlip(p=1),
        A.RandomRotate90(p=1),
        A.HueSaturationValue(hue_shift_limit=27, sat_shift_limit=30, val_shift_limit=20, p=1),
        A.RandomBrightnessContrast(brightness_limit=0.24, contrast_limit=0.2, p=1),
        A.GaussNoise(std_range=(0.05, 0.15), p=1),  # albumentations 2.x: std_range chuẩn hóa 0-1 (thay cho var_limit cũ)
    ], n=3, replace=False, p=1)
], bbox_params=bbox_params)

valid_ext = ('.jpg', '.jpeg', '.png')
# Chỉ augment ẢNH GỐC trong train (bỏ qua file aug_ để chạy lại nhiều lần không bị nhân đôi)
train_files = [f for f in os.listdir(TRAIN_IMG_DIR)
               if f.lower().endswith(valid_ext) and not f.startswith('aug_')]

count_success = 0
for fname in tqdm(train_files, desc="Đang Augment train"):
    base_name = os.path.splitext(fname)[0]
    img_path = os.path.join(TRAIN_IMG_DIR, fname)
    lbl_path = os.path.join(TRAIN_LBL_DIR, base_name + '.txt')

    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV đọc BGR -> đổi sang RGB chuẩn

    bboxes, labels = [], []
    if os.path.exists(lbl_path):  # trích class_id + tọa độ bbox từ file txt
        with open(lbl_path, 'r') as f:
            for line in f:
                p = line.split()
                if len(p) >= 5:  # đủ 5 phần tử mới là bbox hợp lệ
                    labels.append(int(float(p[0])))      # số đầu = class_id
                    bboxes.append([float(x) for x in p[1:5]])  # 4 số: cx, cy, w, h (chuẩn hóa)

    for i in range(2):  # 1 ảnh gốc -> 2 ảnh ảo khác nhau
        new_name = f"aug_{i}_{base_name}"  # đặt tên để truy ngược về ảnh gốc
        try:
            # ~30% ảnh xám (mô phỏng trời tối/mây che), ~70% biến đổi hình học+màu để đa dạng
            if random.random() < 0.3:
                aug = transform_gray(image=img, bboxes=bboxes, class_labels=labels)
            else:
                aug = transform_random(image=img, bboxes=bboxes, class_labels=labels)

            cv2.imwrite(os.path.join(TRAIN_IMG_DIR, new_name + ".jpg"),
                        cv2.cvtColor(aug['image'], cv2.COLOR_RGB2BGR))  # đổi RGB->BGR để ghi file

            with open(os.path.join(TRAIN_LBL_DIR, new_name + ".txt"), 'w') as f:
                for c, b in zip(aug['class_labels'], aug['bboxes']):  # zip để class & bbox không lệch
                    f.write(f"{int(c)} {' '.join([f'{x:.6f}' for x in b])}\n")
            count_success += 1
        except Exception:
            pass

n_train_total = len([f for f in os.listdir(TRAIN_IMG_DIR) if f.lower().endswith(valid_ext)])
print(f"✅ Đã tạo {count_success} ảnh Augment (chỉ trong train).")
print(f"📊 Train sau augment: {n_train_total} ảnh | Val & Test giữ nguyên ảnh gốc.")

# ==========================================
# TẠO FILE dataset.yaml
# ==========================================
print("Tạo file dataset.yaml để train model...")
yaml_content = (
    f"path: {FINAL_SPLIT_DIR}\n"
    f"train: train/images\n"
    f"val: val/images\n"
    f"test: test/images\n\n"
    f"nc: {len(CLASSES)}\n"
    f"names: {CLASSES}\n"
)
with open(os.path.join(FINAL_SPLIT_DIR, 'dataset.yaml'), 'w', encoding='utf-8') as f:
    f.write(yaml_content)
print("✅ Đã tạo dataset.yaml!")
print(f"🎉 DATASET SẴN SÀNG TRAIN TẠI: {FINAL_SPLIT_DIR}")


--- BƯỚC 2: DATA AUGMENTATION (CHỈ TRÊN TẬP TRAIN) ---


Đang Augment train: 100%|██████████| 1423/1423 [01:22<00:00, 17.29it/s]

✅ Đã tạo 2846 ảnh Augment (chỉ trong train).
📊 Train sau augment: 4269 ảnh | Val & Test giữ nguyên ảnh gốc.
Tạo file dataset.yaml để train model...
✅ Đã tạo dataset.yaml!
🎉 DATASET SẴN SÀNG TRAIN TẠI: E:/TIEN_XU_LI_ANH_ANTENNA\Output_files\YOLO_Final_Dataset
